## Agents Skills - part 2

Now we are ready to start adding skills. Let's warm up by re-implementing the Titanic analysis from the Agent tutorial.

The usual requirements:

In [ ]:
!pip -q install ollama python-frontmatter

Since this tutorial is not about the core of the agent, the code from part 1 is provided in 'helpers/convenience.py':

In [ ]:
from helpers.convenience import SkilledAgent, show_messages

The only difference from part 1 is an additional tool to let us render images directly in the notebook:
```python
def inline_image(path: str) -> str:
    """
    Render image at path.

    Args:
        path (str): Path to the image

    Returns:
        str: status message
    """
    print(f"    inline_image: {path}")
    from IPython.display import Image, display
    try:
        img = Image(filename=path)
        display(img)
        return f"Successfully rendered {path}"
    except:
        return "Error: Could not render image"
```

So from here everything should be familiar.

In [ ]:
OLLAMA_HOST = 'http://10.129.20.4:9090/big'
OLLAMA_MODEL='qwen3:14b' 
sa = SkilledAgent(OLLAMA_HOST, OLLAMA_MODEL, 'skills')
print(list(sa.known_tools))
print(list(sa.skills))

Like before, we'll start by working with the Titanic dataset available in 'work/titanic.csv'

In [ ]:
answer = sa.task("The file 'titanic.csv' from the 'work/' directory contains facts about the fate of a subset of the passengers abord the Titanic. Analyse the data and report how many passenger are listed in the data set?.")
print(answer)

This simple task is likely to fail, in testing I got diferent answers (correct is 887). The reason is that we have not provided any guidance as to how to deal with CSV files or go about the analysis. We'll address that now by adding an 'analysing-data' skill. 

### Adding a skill

Create a folder in 'skills/' and name it 'analysing-data'. Create a file 'SKILL.md' in it with the following content:
~~~
---
name: analysing-data
description: Instructions for loading and analysing data. 
---

## Overview

This skill lets you read and analyse data files.

You can use the python package Pandas to read and analyse data files.

ALWAYS use the './work/' directory for all file output.

### How to read a file into a Pandas dataframe

Use the data loading capabilities of pandas to read a file into a dataframe, but beware of the file format.

Example: Loading CSV data

```python
import pandas as pd

df = pd.read_csv('titanic.csv')
```

## Instructions

1. Gather basic information from the data file

Write a first script to read the data into a dataframe and use 'head()' on the dataframe as a preliminary step to find out column labels and get a grip on the data type stored in each column as a reference for future steps.
Run the script using the 'bash' tool to get the desired output.

Example script: 
```python
import pandas as pd

df = pd.read_csv('titanic.csv')
print(df.head())
```

2. Write analysis script

Taking the result of the previous step into account, write a python script to perform the required analysis and reporting and save it as file using the 'write_file' tool.


3. Run the analysis script

Run the analysis script using the 'bash' tool, e.g. bash("python foo.py")

3. Rendering images

If the analysis script generated an image, ALWAYS use the tool 'inline_image' to show it to the user, e.g. inline_image(path="work/foo.png")

~~~

The skill is pretty much self-explanatory, but some points deserve highlighting:

- the "directive" to use Pandas
- the "imperative" to use the 'work/' directory
- the frequent use of examples
- the detailed step-by-step instructions
- the explicit choice of tool in the steps

With that we can start over:

In [ ]:
sa = SkilledAgent(OLLAMA_HOST, OLLAMA_MODEL, 'skills')
list(sa.skills)

N.B. that 'analysing-data' should be listed. If not, make sure the `SKILL.md` file is correctly placed and has the correct content.

In [ ]:
answer = sa.task("The file 'titanic.csv' from the 'work/' directory contains facts about the fate of a subset of the passengers abord the Titanic. Analyse the data and report how many passenger are listed in the data set?.")
print(answer)

In [ ]:
answer = sa.task("What data is recorded for each passenger, and what is the name of the corresponding column?")
print(answer)

In [ ]:
answer = sa.task("What passenger class was most likely to survive?")
print(answer)

In [ ]:
answer = sa.task("Generate a diagram showing likelihood of survival (in percent) versus fare. Bin the fares into suitable ranges, e.g. width of 20 up to 200 and then one bin for all fares above 200.")

To see what is going back and forth, use `show_messages` and examine the files in the 'work/' directory.

In [ ]:
show_messages(sa.messages)

### Dataportal skills

To be continued…